# **Extracting Multi-Year Urban Features Within 15/5-Min Walk Isochrones in Melbourne**
**Note:**
- Uses `graphs_dict_15min.pkl` and `graphs_dict_5min.pkl` from *isochrone_extraction.ipynb*  

**Outputs** (saved under `Output_data_from_urban_form_multiyear_scripts`, prefixed with `15min_` or `5min_`):
- `melbourne_building_counts_df_all_years.csv`
- `melbourne_job_counts_df_all_years_cat.csv`
- `melbourne_total_dwellings.csv`
- `melbourne_parking_counts_df_all_years.csv`

**Processing Steps**

Each output is generated under the corresponging section: 

1. **Building counts** (based on City of Melbourne data)
2. **Jobs counts** (based on City of Melbourne data)
3. **Dwelling counts** (based on City of Melbuorne data)
4. **Off-street car parking counts** (based on City of Melbourne data) <br>

### **IMPORTANT:**  
Modify **`isochrone_type`** to **5** or **15** as needed, and **run all cells before changing it**.  

**Start Date:** *Nov 11, 2024*  
**Last Modification:** *Mar 25, 2025*  
**Authors:** *Benjamin Tarver & Kanaha Shoji*  

In [1]:
# Import necessary libraries and functions
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import pickle
import os

/Users/shoji/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/shoji/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Set base directory
base_dir = '/Users/shoji/Library/CloudStorage/OneDrive-epfl.ch/2025_03_Melbourne_walkability_study_final'
# Set directory to where open data are stored
open_data_dir = os.path.join(base_dir,'Open_datasets')
# set directory for output data
output_dir = os.path.join(base_dir,'Output_data_from_urban_form_multiyear_scripts')

In [3]:
isochrone_type = 15 # change to 15 to generate results for 15 min isochrones

In [4]:
# Load pre-made graphs_dict      
with open(os.path.join(base_dir,f'Output_data_from_isochrone_scripts/graphs_dict_{isochrone_type}min.pkl'), 'rb') as f:
    graphs_dict = pickle.load(f)

In [5]:
years = np.arange(2009,2020).astype(int) #('str')
target_crs = "EPSG:3857"

## **Building counts** (based on City of Melbourne data)
Data can be downloaded from https://data.melbourne.vic.gov.au/explore/dataset/buildings-with-name-age-size-accessibility-and-bicycle-facilities/export/?refine.census_year=2019. 

In [6]:
# New section from this version
building_uses = gpd.read_file(os.path.join(open_data_dir,'buildings-with-name-age-size-accessibility-and-bicycle-facilities.geojson'))
building_uses = building_uses.set_crs("EPSG:4326")
if building_uses.crs is None:
    building_uses.set_crs(epsg=4326, inplace=True)  # Set initial CRS if not set

In [7]:
category_mapping = {
    'Retail': ['Retail - Cars', 'Retail - Shop', 'Retail - Showroom', 'Retail - Stall'],
    'Office': ['Office'],
    'Industry': ['Manufacturing', 'Equipment Installation', 'Storage', 'Workshop/Studio', 'Wholesale'],
    'Service': ['Commercial Accommodation', 'Community Use'],
    'Entertainment': ['Performances, Conferences, Ceremonies', 'Entertainment/Recreation - Indoor'],
    'Education': ['Educational/Research'],
    'Health care': ['Hospital/Clinic'],
    'Public Administration': ['Public Display Area', 'Transport'],
    'Household': ['Residential Apartment', 'House/Townhouse', 'Student Accommodation', 'Institutional Accommodation'],
    'Unused':['Unoccupied - Unused','Unoccupied - Under Renovation', 'Unoccupied - Under Construction', 'Unoccupied - Under Demolition/Condemned'],
    'Parking':['Parking - Commercial Covered', 'Parking - Private Covered']
}

In [8]:
# Re-project  to the target CRS
building_uses = building_uses.to_crs(target_crs)
building_uses['census_year'] = building_uses['census_year'].astype(int)
building_all_years = []

# Iterate over each polygon in the graphs_dict
for polygon_id, polygon in graphs_dict.items():
    # Extract and re-project the polygon geometry once
    polygon = polygon['geometry']

    # Convert polygon to GeoDataFrame to enable CRS conversion
    polygon_gdf = gpd.GeoDataFrame(geometry=[polygon], crs='EPSG:4326')
    polygon_gdf = polygon_gdf.to_crs(target_crs)
    polygon = polygon_gdf.geometry.iloc[0]

    # Loop through each census year
    for year in years:
        # Filter buildings within the polygon and for the current census year
        buildings_within_polygon = building_uses[
            (building_uses.geometry.intersects(polygon)) &
            (building_uses['census_year'] == year)
        ]
        # Initialize a dictionary to hold area and count results for the current polygon and year
        count_results = {'polygon_id': polygon_id, 'census_year': year}
        
        # Loop through each category in the mapping
        for category in category_mapping.keys():
            # Filter buildings that fall into the current category
            category_buildings = buildings_within_polygon[
                buildings_within_polygon['predominant_space_use'].isin(category_mapping[category])
            ]     
            # Calculate the total count for the current category
            total_count = category_buildings.shape[0]  # Count of buildings in this category
            # Store results in the dictionary
            count_results[f'{category}_count'] = total_count
            
        # Append the results for the current polygon and year
        building_all_years.append(count_results)

# Convert results to a DataFrame for easier analysis
building_df_all_years = pd.DataFrame(building_all_years)

In [9]:
print(building_df_all_years.head())

  polygon_id  census_year  Retail_count  Office_count  Industry_count  \
0      G_000         2009           154           354              76   
1      G_000         2010           153           354              76   
2      G_000         2011           147           353              77   
3      G_000         2012           143           352              69   
4      G_000         2013           136           346              70   

   Service_count  Entertainment_count  Education_count  Health care_count  \
0             69                  251               67                  8   
1             69                  254               69                  8   
2             70                  265               67                  8   
3             70                  272               61                  8   
4             69                  272               59                  8   

   Public Administration_count  Household_count  Unused_count  Parking_count  
0                  

## **Job counts** (based on City of Melbourne data)
'Jobs per space use for blocks':https://data.melbourne.vic.gov.au/explore/dataset/employment-by-block-by-space-use/information/?disjunctive.block_id&disjunctive.total_jobs_in_block <br>
'Blocks for Census of Land Use and Employment (CLUE)': https://data.melbourne.vic.gov.au/explore/dataset/blocks-for-census-of-land-use-and-employment-clue/information/



In [10]:
# Import employment GeoDFs
clue_blocks = gpd.read_file(os.path.join(open_data_dir,'blocks-for-census-of-land-use-and-employment-clue.geojson')).to_crs("EPSG:4326")
clue_jobs = pd.read_csv(os.path.join(open_data_dir,'employment-by-block-by-space-use.csv'))

clue_blocks = clue_blocks.rename(columns={'block_id': 'Block ID'})

In [11]:
# Define the categorization mapping
category_mapping = {
    'Retail': ['Retail - Cars', 'Retail - Shop', 'Retail - Showroom', 'Retail - Stall'],
    'Office': ['Office'],
    'Industry': ['Manufacturing', 'Transport', 'Transport/Storage - Uncovered', 'Wholesale', 'Workshop/Studio',
                 'Unoccupied - Under Construction','Unoccupied - Under Demolition/Condemned',
                 'Unoccupied - Under Renovation','Unoccupied - Undeveloped Site','Unoccupied - Unused'],
    'Service': ['Commercial Accommodation', 'Community Use', 'Equipment Installation', 'Parking - Commercial Covered',
                'Parking - Commercial Uncovered', 'Parking - Private Covered', 'Parking - Private Uncovered',
                'House/Townhouse','Student Accommodation','Storage'],
    'Entertainment': ['Entertainment/Recreation - Indoor', 'Performances, Conferences, Ceremonies', 
                      'Sports and Recreation - Outdoor'],
    'Education': ['Educational/Research'],
    'Health care': ['Hospital/Clinic'],
    'Public Administration': ['Institutional Accommodation', 'Public Display Area', 'Square/Promenade', 'Park/Reserve']
}

In [12]:
jobs_all_cat_dfs = []

for year in years:
    # Perform the merge
    merged_df = clue_blocks.merge(clue_jobs.loc[clue_jobs['Census year']==int(year)], on='Block ID')

    # Convert the merged DataFrame to a GeoDataFrame
    clue_gdf = gpd.GeoDataFrame(merged_df, geometry='geometry')

    # Initialize an empty list to store the results
    df_jobs = []

    for key, value in graphs_dict.items():
        polygon = value['geometry']
        # Convert polygon to GeoDF
        poly_gdf = gpd.GeoDataFrame([1], geometry=[polygon], crs="EPSG:4326")
        poly_gdf['preserved_geom'] = poly_gdf.geometry

        # Get the jobs within the polygon, sum
        job_iter = poly_gdf.sjoin(clue_gdf, predicate='intersects')
        job_iter['Polygon'] = key
        job_iter = job_iter.drop(columns=['preserved_geom', 'geometry', 'geo_point_2d'])

        # Initialize a dictionary to store the sum of jobs for each category
        category_sums = {category: 0 for category in category_mapping.keys()}

        # Sum the jobs for each category
        for category, columns in category_mapping.items():
            category_sums[category] = job_iter[columns].sum(axis=1).sum()
        # Get the total jobs in block
        total_jobs = job_iter['Total jobs in block'].sum()
        # Append the results (key, totals for each category, total jobs, and year)
        df_jobs.append([key, *category_sums.values(), total_jobs, year])

    # Create a DataFrame with the results
    columns = ['Polygon'] + list(category_sums.keys()) + ['Total Jobs in Block', 'Usable Year']
    job_counts_df = pd.DataFrame(df_jobs, columns=columns)

    # Append the results for this year to the overall list
    jobs_all_cat_dfs.append(job_counts_df)

final_jobs_cat_df = pd.concat(jobs_all_cat_dfs, ignore_index=True)

In [13]:
print(final_jobs_cat_df.head())

  Polygon   Retail    Office  Industry  Service  Entertainment  Education  \
0   G_000  16411.0  177392.0     921.0   1820.0        14073.0     7497.0   
1   G_001  16845.0  182075.0     921.0   1953.0        21358.0     7497.0   
2   G_002  17062.0  148247.0     905.0   1506.0        19030.0     2975.0   
3   G_003    232.0   21816.0      18.0     53.0          377.0      101.0   
4   G_004   3131.0   81706.0     617.0    538.0         3941.0      779.0   

   Health care  Public Administration  Total Jobs in Block  Usable Year  
0       1415.0                    7.0             232921.0         2009  
1       1616.0                   38.0             247950.0         2009  
2       1404.0                   38.0             202226.0         2009  
3         15.0                    0.0              23682.0         2009  
4        128.0                    0.0              95729.0         2009  


## **Dweling counts** (basedon City of Melbourne data)
Data can be downloaded from https://data.melbourne.vic.gov.au/explore/dataset/residential-dwellings/information/?disjunctive.dwelling_type&disjunctive.clue_small_area&disjunctive.block_id&disjunctive.dwelling_number

In [14]:
# Load the residences CSV
residences_df = pd.read_csv(os.path.join(open_data_dir, 'residential-dwellings.csv'))

# Separate the data into two DataFrames: one with coordinates, one without
residences_with_coords = residences_df.dropna(subset=['longitude', 'latitude'])
residences_without_coords = residences_df[residences_df[['longitude', 'latitude']].isnull()]

# Convert the residences with coordinates into a GeoDataFrame using longitude and latitude

residences_with_coords.loc[:, 'geometry'] = residences_with_coords.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
residences_gdf_with_coords = gpd.GeoDataFrame(residences_with_coords, geometry='geometry', crs="EPSG:4326")

# Initialize an empty list to store the results for all years
dwellings_all_cat_dfs = []

# Iterate through the years you are interested in
for year in years:
    # Filter residences for the current year
    residences_year_with_coords = residences_gdf_with_coords[residences_gdf_with_coords['census_year'] == int(year)]
    residences_year_without_coords = residences_without_coords[residences_without_coords['census_year'] == int(year)]

    df_dwellings = []
    # Iterate through the polygons in graphs_dict
    for key, value in graphs_dict.items():
        polygon = value['geometry']

        # Convert the polygon to a GeoDataFrame
        poly_gdf = gpd.GeoDataFrame([1], geometry=[polygon], crs="EPSG:4326")
        poly_gdf['preserved_geom'] = poly_gdf.geometry

        # Handle Residences with Coordinates (spatial join)
        dwellings_within_polygon = poly_gdf.sjoin(residences_year_with_coords, predicate='intersects')
        total_dwellings_with_coords = dwellings_within_polygon['dwelling_number'].sum()

        # Handle Residences without Coordinates (use block_id)
        # Merge residences without coordinates with clue_blocks
        merged_without_coords = residences_year_without_coords.merge(clue_blocks, left_on='block_id', right_on='Block ID', how='inner')

        # Convert the merged DataFrame to a GeoDataFrame
        clue_gdf_without_coords = gpd.GeoDataFrame(merged_without_coords, geometry='geometry', crs="EPSG:4326")

        # Perform a spatial join with the polygon
        dwellings_in_block_polygon = poly_gdf.sjoin(clue_gdf_without_coords, predicate='intersects')
        total_dwellings_without_coords = dwellings_in_block_polygon['dwelling_number'].sum()

        # Append the combined result (key, total dwellings, and year)
        total_dwellings = total_dwellings_with_coords + total_dwellings_without_coords
        df_dwellings.append([key, total_dwellings, year])

    # Create a DataFrame with the results for this year
    columns = ['Polygon', 'Total Dwellings', 'Usable Year']
    dwellings_counts_df = pd.DataFrame(df_dwellings, columns=columns)

    # Append the results for this year to the overall list
    dwellings_all_cat_dfs.append(dwellings_counts_df)

# Concatenate all results into a single DataFrame
total_dwellings_df = pd.concat(dwellings_all_cat_dfs, ignore_index=True)

/var/folders/b9/8tykx59n0x9d083w665r66g00000gq/T/ipykernel_22627/3478870415.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  residences_with_coords.loc[:, 'geometry'] = residences_with_coords.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)


## **Off-street car parking counts** (based on City of Melbourne data)
Count available parking lot spaces in each polygon.<br>
Data can be downloaded from https://data.melbourne.vic.gov.au/explore/dataset/off-street-car-parks-with-capacity-and-type/export/?dataChart=eyJxdWVyaWVzIjpbeyJjaGFydHMiOlt7InR5cGUiOiJjb2x1bW4iLCJmdW5jIjoiU1VNIiwieUF4aXMiOiJwYXJraW5nX3NwYWNlcyIsInNjaWVudGlmaWNEaXNwbGF5Ijp0cnVlLCJjb2xvciI6InJhbmdlLWN1c3RvbSJ9XSwieEF4aXMiOiJjZW5zdXNfeWVhciIsIm1heHBvaW50cyI6IiIsInRpbWVzY2FsZSI6InllYXIiLCJzb3J0IjoiIiwic2VyaWVzQnJlYWtkb3duIjoicGFya2luZ190eXBlIiwic3RhY2tlZCI6Im5vcm1hbCIsImNvbmZpZyI6eyJkYXRhc2V0Ijoib2ZmLXN0cmVldC1jYXItcGFya3Mtd2l0aC1jYXBhY2l0eS1hbmQtdHlwZSIsIm9wdGlvbnMiOnt9fX1dLCJkaXNwbGF5TGVnZW5kIjp0cnVlLCJhbGlnbk1vbnRoIjp0cnVlLCJ0aW1lc2NhbGUiOiIifQ%3D%3D

In [15]:
# Define a function to calculate the counts of parking lot spaces for each polygon
def calculate_parking_counts(polygon, parking):

    # Convert polygon to GeoDF
    poly_gdf = gpd.GeoDataFrame([1], geometry=[polygon], crs="EPSG:4326")

    # Get the dwelling within the polygon
    parking_gdf = poly_gdf.sjoin(parking, predicate='covers')

    # Drop duplicates to ensure we only have unique dwelling with their counts
    parking_gdf = parking_gdf.drop_duplicates()

    # Group by 'Polygon' and calculate the sum of dwelling type
    returnable = parking_gdf.groupby('parking_type', as_index=False)['parking_spaces'].sum()

    # Sum space counts for polygon
    returnable.loc[len(returnable), ['parking_type', 'parking_spaces']] = 'All', sum(returnable['parking_spaces'])
    return returnable

In [16]:
# Import car parking DF
parking_gdf = gpd.read_file(os.path.join(open_data_dir,'off-street-car-parks-with-capacity-and-type.geojson')).to_crs("EPSG:4326")

# Get counts of number of parking spaces per polygon
parking_counts_all_yrs = []

for year in years:
    # Initialize an empty DF to store the results
    parking_counts_df = pd.DataFrame(columns = ['Polygon', 'parking_type', 'parking_spaces'])

    for key, value in graphs_dict.items():
        polygon = value['geometry']
        # Convert the 'census_year' column in parking_gdf to integers
        parking_gdf['census_year'] = parking_gdf['census_year'].astype(int)
        # Call calculate parking counts function
        df_iter = calculate_parking_counts(polygon, parking_gdf.loc[parking_gdf['census_year']==year])
        df_iter['Polygon'] = key
        # Remove empty columns from both DataFrames before concatenation
        df_iter = df_iter.dropna(axis=1, how='all')
        parking_counts_df = parking_counts_df.dropna(axis=1, how='all')
        # Concatenate iterator DF with output DF
        parking_counts_df = pd.concat([parking_counts_df, df_iter], ignore_index=True)

    # I acknowledge this is terrible code, but if it ain't broke don't fix it!
    # Regroup columns for usefulness in feature analysis
    parking_counts_df['Commercial Parking Spaces'] = parking_counts_df.apply(lambda row: row['parking_spaces'] if row['parking_type'] == 'Commercial' else None, axis=1)
    parking_counts_df['Private Parking Spaces'] = parking_counts_df.apply(lambda row: row['parking_spaces'] if row['parking_type'] == 'Private' else None, axis=1)
    parking_counts_df['Residential Parking Spaces'] = parking_counts_df.apply(lambda row: row['parking_spaces'] if row['parking_type'] == 'Residential' else None, axis=1)
    parking_counts_df['Total Parking Spaces'] = parking_counts_df.apply(lambda row: row['parking_spaces'] if row['parking_type'] == 'All' else None, axis=1)
    parking_counts_df = parking_counts_df.drop(columns=['parking_type','parking_spaces'])
    parking_counts_df = parking_counts_df.groupby('Polygon').agg({'Residential Parking Spaces': 'sum', 
                                                                  'Private Parking Spaces': 'sum', 
                                                                  'Commercial Parking Spaces': 'sum', 
                                                                  'Total Parking Spaces': 'sum'}).reset_index()
    parking_counts_df['Usable Year'] = year

    parking_counts_all_yrs.append(parking_counts_df)
# Concatenate all results into a single DataFrame
total_parking_counts_all_yrs = pd.concat(parking_counts_all_yrs, ignore_index=True)
print(total_parking_counts_all_yrs.head())

  Polygon Residential Parking Spaces Private Parking Spaces  \
0   G_000                          0                      0   
1   G_001                          0                      0   
2   G_002                          0                      0   
3   G_003                          0                      0   
4   G_004                          0                      0   

  Commercial Parking Spaces  Total Parking Spaces  Usable Year  
0                         0                   0.0         2009  
1                         0                   0.0         2009  
2                         0                   0.0         2009  
3                         0                   0.0         2009  
4                         0                   0.0         2009  


In [17]:
# Check the dataframe created
print('Building counts')
print(building_df_all_years.head())
print('Jobs based on space use')
print(final_jobs_cat_df.head())
print('Dwellings')
print(total_dwellings_df.head())
print('Parking count')
print(total_parking_counts_all_yrs.tail())

Building counts
  polygon_id  census_year  Retail_count  Office_count  Industry_count  \
0      G_000         2009           154           354              76   
1      G_000         2010           153           354              76   
2      G_000         2011           147           353              77   
3      G_000         2012           143           352              69   
4      G_000         2013           136           346              70   

   Service_count  Entertainment_count  Education_count  Health care_count  \
0             69                  251               67                  8   
1             69                  254               69                  8   
2             70                  265               67                  8   
3             70                  272               61                  8   
4             69                  272               59                  8   

   Public Administration_count  Household_count  Unused_count  Parking_count  
0  

In [ ]:
# Save the final results to a CSV
building_df_all_years.to_csv(os.path.join(output_dir,f'{isochrone_type}min_melbourne_building_counts_df_all_years.csv'),index=False)
final_jobs_cat_df.to_csv(os.path.join(output_dir,f'{isochrone_type}min_melbourne_job_counts_df_all_years_cat.csv'),index=False)
total_dwellings_df.to_csv(os.path.join(output_dir, f'{isochrone_type}min_melbourne_total_dwellings.csv'),index=False)
total_parking_counts_all_yrs.to_csv(os.path.join(output_dir,f'{isochrone_type}min_melbourne_parking_counts_df_all_years.csv'),index=False) 